# Day 13 - Transformers & BERT: Text Intelligence

## RetailMind AI Increment

Day 13 upgrades the Day-12 TF-IDF baseline with a pretrained Transformer backbone for two connected tasks: Amazon-review sentiment and support-ticket routing. The project fine-tunes `distilbert-base-uncased`, reports per-class metrics, compares sentiment performance with Day 12, and persists deployable model/tokenizer folders.

### Business objective
Contextual language understanding helps distinguish phrases such as “not bad” from negative wording and routes support tickets to the correct team. The upgrade is justified only if measurable quality gains outweigh greater latency, memory, and operational cost.

## Data and environment

Local files only:

- `data/amazon_reviews.csv`: `text,label` or `title,content,label`.
- `data/support_tickets.csv`: the same text schema with routing labels such as `refund`, `delivery`, `damaged_item`, or `account_support`.

The first run needs access to the pretrained Hugging Face model unless the model is already cached. Fine-tuning uses the PyTorch backend installed with Transformers.

In [ ]:
import sys
from pathlib import Path
repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(repo_root / "src"))
import day13_bert_text_intelligence as day13

print("Backbone:", day13.MODEL_NAME)
print("Maximum tokens:", day13.MAX_LENGTH)
print("Epochs:", day13.EPOCHS)

## 1. Load and audit both classification tasks

Sentiment labels are normalized to binary IDs. Ticket categories remain named classes and receive deterministic IDs. Stratified splitting keeps every class represented in evaluation.

In [ ]:
reviews = day13.read_csv_records(repo_root / day13.REVIEWS_PATH, task="sentiment")
tickets = day13.read_csv_records(repo_root / day13.TICKETS_PATH, task="routing")

from collections import Counter
print("Review records:", len(reviews), Counter(label for _, label in reviews))
print("Ticket records:", len(tickets), Counter(label for _, label in tickets))
print("Example ticket:", tickets[0])

## 2. Tokenization

The pretrained tokenizer adds special tokens, converts subwords to vocabulary IDs, creates attention masks, truncates at 192 tokens, and dynamically pads each batch. Dynamic padding avoids paying the full 192-token cost for every short review.

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(day13.MODEL_NAME)
example = tokenizer(
    reviews[0][0], truncation=True, max_length=day13.MAX_LENGTH,
    return_attention_mask=True
)
print("Input IDs:", example["input_ids"][:20])
print("Tokens:", tokenizer.convert_ids_to_tokens(example["input_ids"][:20]))
print("Sequence length:", len(example["input_ids"]))

## 3. Fine-tune BERT for review sentiment

`AutoModelForSequenceClassification` adds a binary classification head to DistilBERT. The default learning rate is `2e-5`, batch size is 8, and the notebook uses two epochs for a CPU-conscious demonstration. The best checkpoint is selected by macro-F1 rather than training loss.

In [ ]:
# Run when the pretrained backbone is available/cached.
# sentiment_metrics = day13.train_classifier(
#     reviews, str(repo_root / day13.SENTIMENT_DIR), "sentiment"
# )
# sentiment_metrics

## 4. Compare BERT with Day 12

The comparison must use the same label meaning and preferably the same untouched examples. Day 12 reports classical F1; Day 13 reports macro-F1 so both classes contribute equally. Record the absolute lift and balance it against training time, model size, and inference latency.

In [ ]:
day12_metrics = day13.load_day12_metrics(repo_root / day13.DAY12_METRICS_PATH)
print("Day 12 metrics:", day12_metrics)
# After training:
# bert_f1 = sentiment_metrics["eval_macro_f1"]
# print("BERT F1 lift:", bert_f1 - day12_metrics["f1"])

## 5. Fine-tune the same backbone for ticket routing

Ticket routing is a separate multi-class task with its own classification head and label mapping. Reusing the architecture does not mean reusing the sentiment head. Each saved model folder includes tokenizer files, configuration, model weights, and the ticket label mapping.

In [ ]:
# routing_metrics = day13.train_classifier(
#     tickets, str(repo_root / day13.ROUTER_DIR), "ticket routing"
# )
# routing_metrics

## 6. Per-class evaluation

Overall accuracy can hide a weak route. For every ticket class, inspect precision, recall, and F1. Low recall means tickets belonging to that route are being missed. Low precision means another team receives too many incorrectly routed tickets. Macro-F1 gives each route equal weight regardless of volume.

In [ ]:
# Example after routing training:
# for key, value in routing_metrics.items():
#     if "class_" in key or key in {"eval_accuracy", "eval_macro_f1"}:
#         print(f"{key:30s} {value:.4f}")

## 7. Inference and confidence-aware routing

A ticket router should not force low-confidence predictions into automation. A practical policy is automatic routing above a validated threshold such as 0.80 and manual triage below it. The threshold must be calibrated on validation data rather than selected arbitrarily.

In [ ]:
# Run after the routing model is trained:
# result = day13.route_ticket(
#     "My package arrived damaged and I need a replacement",
#     repo_root / day13.ROUTER_DIR,
# )
# print(result)
# if result["confidence"] < 0.80:
#     print("Send to manual triage")

## 8. Pipeline replacement strategy

Do not delete the Day-12 baseline. Keep it as a low-cost fallback and benchmark. Replace the active sentiment path with BERT only after BERT demonstrates repeatable lift on the same test data and meets latency/resource limits. Log model version, confidence, predicted label, and routing outcome so production errors can be audited.

## 9. Limitations and next iteration

BERT may inherit social and product-category biases from pretraining and fine-tuning data. Truncation can remove decisive text late in a long review. Confidence is not guaranteed to be calibrated. Domain drift can reduce routing accuracy as new issue types appear. Day 14 can add LLM review summaries and a support chatbot while retaining these classifiers as deterministic guardrails.